# Train the starting models for the recovery pilot

This notebook prepares study checkpoints. It does **not** perform recovery training or answer whether forgotten knowledge remains.

Start with a **CPU runtime** and `MODE = "CHECK"`. Then use `PREPARE` on CPU to preserve the image manifest and protocol in R2. GPU training is a separate, explicit step: choose an A100, set `MODE = "TRAIN"`, and set `RUN_TRAINING = True`. Run one seed and one stage per session. Do not run all three seeds automatically.

## The comparisons

- **Target X = task 3:** the skill to learn and then unlearn.
- **Unseen control Y = task 2:** excluded from all initial learning. A later recovery notebook will probe Y to measure general transfer.
- **Shared stage:** learn task 0 once. Both branches restore this exact checkpoint, so chunk embeddings begin identical and stay frozen afterward.
- **Unlearned branch:** learn 0, 3, 9, 5, 17, 1, 7, 14; then unlearn 3.
- **Never-learned branch:** learn 0, 9, 5, 17, 1, 7, 14. Never read task-3 training images.

Learning 0 before 3 is a deliberate change from the previous diagnostic. It avoids learning the shared chunk embeddings on different first tasks. The branches share retained-task order and starting state; their later weights will differ because one learned task 3. They are not an exact reproduction of the paper.

Task 3 uses 400 images per class from notebook 16. Its 100 reserved images per class are excluded from initial training. Retained tasks use their existing 500 training images per class. Validation remains separate. Existing all-images checkpoints are not loaded.

## The readiness check

After U3, require target accuracy at most 12%, target accuracy before U3 at least 25%, retained mean accuracy above 40%, and spill below 3 percentage points. Spill is the **sum of absolute accuracy changes** across the seven retained tasks. It is not their mean loss. These are practical proposal criteria, not proof of erasure. Failure is recorded and stops advancement to recovery; no automatic parameter search follows.

## Cell 1: Load a fixed code revision

In Colab, clone a separate clean checkout and install only the small storage dependency if missing. Colab normally provides PyTorch, torchvision, NumPy, and Pillow. Do not upgrade them automatically: record the actual runtime versions instead.

The checkout is fixed so later sessions use the same code. On a local machine, use a clean checkout of this revision. This protects your unrelated training edits. No model is constructed here.

In [ ]:
from pathlib import Path
import sys, os, json, hashlib, subprocess, importlib.util
IN_COLAB = "google.colab" in sys.modules
CODE_VERSION = "963349813127151c4b5074dca687dd8392fba4dd"
assert not any(n == "uncle" or n.startswith("uncle.") for n in sys.modules), "Restart the runtime before reloading code."
if IN_COLAB:
    REPO = Path("/content/cark-recovery-study")
    if not REPO.exists():
        subprocess.run(["git", "clone", "https://github.com/sumitasthana/CARK.git", str(REPO)], check=True)
    if subprocess.check_output(["git", "-C", str(REPO), "status", "--porcelain", "--untracked-files=no"], text=True).strip():
        raise RuntimeError("Checkout has local edits. Use a fresh runtime.")
    subprocess.run(["git", "-C", str(REPO), "fetch", "origin"], check=True)
    subprocess.run(["git", "-C", str(REPO), "checkout", "--detach", CODE_VERSION], check=True)
else:
    REPO = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "uncle/task_partition.json").exists()), None)
    if REPO is None:
        raise FileNotFoundError("Open from a clean CARK checkout.")
    actual = subprocess.check_output(["git", "-C", str(REPO), "rev-parse", "HEAD"], text=True).strip()
    dirty = subprocess.check_output(["git", "-C", str(REPO), "status", "--porcelain", "--", "uncle"], text=True).strip()
    if actual != CODE_VERSION or dirty:
        raise RuntimeError("Use a clean checkout of CODE_VERSION, or open this notebook in Colab.")
if importlib.util.find_spec("boto3") is None and IN_COLAB:
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "boto3>=1.36"], check=True)
sys.path.insert(0, str(REPO))
from uncle.config import Config
from uncle.recovery_setup import validate_manifest, build_study_tasks, run_setup_stage
from uncle.storage import R2Store
import torch
print("Fixed code:", CODE_VERSION)
print("PyTorch:", torch.__version__, "CUDA build:", torch.version.cuda)


## Cell 2: Choose the mode, seed, and stage

`CHECK` verifies your notebook-16 manifest locally, with no cloud writes. `PREPARE` saves that manifest and the fixed protocol to R2 on CPU. Run PREPARE once before any GPU stage.

For TRAIN, choose `shared`, then `never_learned`, then `unlearned`, using the same seed and STUDY_ID. Repeat those stages for seeds 1 and 2 after reviewing seed 0. The shared lesson is trained once per seed and reused. A completed stage is verified without retraining. An interrupted lesson restarts from the last completed lesson with a fresh optimizer; it does not resume a partially completed epoch.

The learning settings below are diagnostic choices inherited from the later runs, not the paper's complete settings. The protocol becomes fixed in R2. Changing it requires a new study ID. This cell refuses CPU training of the full ResNet50 model.

In [ ]:
MODE = "CHECK"  # CHECK, PREPARE, or TRAIN
RUN_TRAINING = False
STUDY_ID = "recovery_task3_reserved_v1"
SEED = 0  # one of 0, 1, 2
STAGE = "shared"  # shared, never_learned, or unlearned
BUCKET = "uncle-experiments"
MANIFEST_PATH = Path("/content/manifest.json") if IN_COLAB else REPO / "outputs/recovery_setup/task3_split_v1/manifest.json"
DATA_ROOT = Path("/content/data/tiny-imagenet-200") if IN_COLAB else REPO / "data/tiny-imagenet-200"
DOWNLOAD_DATA = False  # enable only when TRAIN lacks the dataset
RELEASE_GPU_WHEN_DONE = True
TARGET = "3"
CONTROL_Y = "2"
RETAINED = ("0", "9", "5", "17", "1", "7", "14")
assert MODE in ("CHECK", "PREPARE", "TRAIN")
assert SEED in (0, 1, 2) and STAGE in ("shared", "never_learned", "unlearned")
assert STUDY_ID and all(c.isalnum() or c in "_-" for c in STUDY_ID)
if MODE != "TRAIN" and torch.cuda.is_available():
    raise RuntimeError("Use a CPU runtime for CHECK and PREPARE to avoid occupying a GPU.")
if MODE == "TRAIN" and (not RUN_TRAINING or not torch.cuda.is_available()):
    raise RuntimeError("TRAIN requires explicit RUN_TRAINING=True and a GPU runtime.")
PROTOCOL = {"version": 1, "code_version": CODE_VERSION, "target": TARGET,
    "control_y": CONTROL_Y, "retained": list(RETAINED), "seeds": [0, 1, 2],
    "shared_first_task": "0", "target_train_per_class": 400,
    "retained_train_per_class": 500, "initialization": "hyperfan_in",
    "backbone": "resnet50", "epochs": 5, "batch_size": 64,
    "eval_batch_size": 256, "learning_rate": 0.0001, "beta": 0.1,
    "gamma": 0.01, "noise_samples": 10, "burn_in": 100,
    "gate": {"retained_mean_gt": 40, "spill_lt": 3, "target_after_le": 12, "target_before_ge": 25}}
PREFIX = "uncle/recovery_study/" + STUDY_ID
print("Mode:", MODE, "Seed:", SEED, "Stage:", STAGE)
print("No recovery adaptation is included in this notebook.")


## Cell 3: Connect R2 only when needed

CHECK skips this cell. PREPARE and TRAIN use the same existing bucket and secret names as notebook 15. In Colab, enable access to `R2_ENDPOINT`, `R2_ACCESS_KEY_ID`, and `R2_SECRET_ACCESS_KEY` in the Secrets panel. On your computer, supply them as environment variables. Secret values are never printed or saved.

This notebook creates only files under its study prefix. It does not copy or delete older experiments. R2 upload verification reads files back, so checkpoint transfers take time.

In [ ]:
store = None
if MODE in ("PREPARE", "TRAIN"):
    def secret(name):
        value = os.environ.get(name)
        if not value and IN_COLAB:
            from google.colab import userdata
            value = userdata.get(name)
        if not value:
            raise RuntimeError("Missing secret: " + name)
        return value
    store = R2Store.connect(secret("R2_ENDPOINT"), BUCKET,
                            secret("R2_ACCESS_KEY_ID"), secret("R2_SECRET_ACCESS_KEY"))
    print("Connected to bucket:", BUCKET)
else:
    print("CHECK: no R2 connection.")


## Cell 4: Verify and preserve the image manifest

For CHECK or PREPARE in Colab, upload the `manifest.json` from notebook 16 if it is missing. Extract it from the small ZIP you downloaded. This uploads only image lists, not dataset images.

TRAIN retrieves the verified manifest and protocol from R2. It refuses a protocol mismatch before constructing a model. The partition fingerprint must match the saved class mapping. Every selected path and budget is checked. File existence is checked later when datasets are built.

In [ ]:
CACHE = REPO / "outputs/recovery_study_cache"
CACHE.mkdir(parents=True, exist_ok=True)
if MODE == "TRAIN":
    MANIFEST_PATH = store.download(PREFIX + "/manifest.json", CACHE)
    saved_protocol = json.loads(store.download(PREFIX + "/protocol.json", CACHE).read_text())
    if saved_protocol != PROTOCOL:
        raise ValueError("Saved protocol differs. Restore the original settings or use a new STUDY_ID.")
else:
    if not MANIFEST_PATH.exists() and IN_COLAB:
        from google.colab import files
        uploaded = files.upload()
        if "manifest.json" not in uploaded:
            raise ValueError("Upload the manifest.json from notebook 16.")
        MANIFEST_PATH.write_bytes(uploaded["manifest.json"])
    if not MANIFEST_PATH.exists():
        raise FileNotFoundError("Run notebook 16 and supply its manifest.json.")
manifest = json.loads(MANIFEST_PATH.read_text())
if str(manifest["task_id"]) != TARGET:
    raise ValueError("This protocol requires the task-3 manifest.")
validate_manifest(manifest, REPO / "uncle/task_partition.json")
MANIFEST_HASH = hashlib.sha256(MANIFEST_PATH.read_bytes()).hexdigest()
print("Manifest SHA-256:", MANIFEST_HASH)
print("PASS: 4,000 initial task-3 images; 1,000 reserved; 500 validation.")
if MODE == "PREPARE":
    protocol_path = CACHE / "protocol.json"
    protocol_path.write_text(json.dumps(PROTOCOL, indent=2) + "\n")
    store.upload(protocol_path, PREFIX + "/protocol.json")
    store.upload(MANIFEST_PATH, PREFIX + "/manifest.json")
    print("Protocol and manifest verified in R2. No dataset or models uploaded.")


## Cell 5: Review compute and storage before training

All eight tasks are indexed only in TRAIN. No model is built during CPU preparation. The shared stage trains one lesson. The reference adds six lessons, and the unlearned branch adds seven lessons plus 100 unlearning updates. Together, that is 14 learning lessons per seed, not 16, because the shared lesson is reused.

At five epochs, task 3 uses 315 optimizer updates (4,000 images, batch size 64); each retained lesson uses 395. That is 5,450 learning updates per seed plus 100 unlearning updates. Runtime is not yet measured for this reserved-data design. Start with seed 0 and review its timing before committing more GPU sessions.

Keep four checkpoints per seed: shared, never-learned final, before-unlearning, and after-unlearning. Historical checkpoints were about 537 MB each; use that only as a rough storage estimate (about 2.15 GB per seed). Rolling checkpoints overwrite only the current study stage's progress file. Reports keep completed lesson measurements. Each completed lesson transfers a checkpoint to R2; each epoch transfers only a small report.

Evaluation runs once per learning epoch, and at unlearning steps 0, 1, 20, 40, 60, 80, and 100. No per-update tensor hashing or gradient attribution is performed.

In [ ]:
if MODE == "TRAIN":
    print("GPU:", torch.cuda.get_device_name(0))
    print("Selected stage:", STAGE)
else:
    print("CPU setup complete. Use PREPARE to preserve the protocol, then choose a separate GPU session.")


## Cell 6: Build datasets on local disk (TRAIN only)

The dataset is downloaded only if missing and DOWNLOAD_DATA is enabled. Downloading and indexing are CPU and disk operations even in a GPU session. Reuse an existing local dataset when available; a fresh Colab session may require another download. The notebook does not upload Tiny ImageNet to R2.

The loader replaces task 3's ordinary training data with the saved 4,000-image list. Recovery images are never supplied to the trainer. Task Y is not built. Other tasks retain their usual training data. The never-learned branch omits task 3 entirely from its trainer datasets.

In [ ]:
tasks = None
if MODE == "TRAIN":
    from uncle.tinyimagenet import prepare_data
    DATA_ROOT = prepare_data(DATA_ROOT, download=DOWNLOAD_DATA)
    tasks = build_study_tasks(DATA_ROOT, manifest, REPO / "uncle/task_partition.json", RETAINED, CONTROL_Y)
    if STAGE == "shared":
        tasks = {"0": tasks["0"]}
    elif STAGE == "never_learned":
        tasks.pop(TARGET)
    assert CONTROL_Y not in tasks
    print("Training dataset sizes:", {t: len(d["train"]) for t, d in tasks.items()})
else:
    print("Skipped dataset loading.")


## Cell 7: Restore the stage's last completed lesson (TRAIN only)

Both branches download the same seed-specific shared checkpoint. If the chosen stage already has progress, download its settings and latest checkpoint and verify their hashes. An incompatible configuration is refused.

Use only one session to write a stage at a time. Keep the same study ID and seed when resuming an interrupted lesson. Do not point this notebook at older experiment checkpoints.

In [ ]:
source = None
OUTPUT = REPO / "outputs/recovery_study" / STUDY_ID / f"seed_{SEED}" / STAGE
STAGE_PREFIX = PREFIX + f"/seed_{SEED}/" + STAGE
if MODE == "TRAIN":
    OUTPUT.mkdir(parents=True, exist_ok=True)
    if STAGE != "shared":
        source = store.download(PREFIX + f"/seed_{SEED}/shared/checkpoint.pt", CACHE)
    for name in ("settings.json", "checkpoint.pt", "before_unlearning.pt"):
        key = STAGE_PREFIX + "/" + name
        if store.info(key) is not None:
            remote = store.download(key, CACHE)
            local = OUTPUT / name
            if local.exists() and hashlib.sha256(local.read_bytes()).hexdigest() != hashlib.sha256(remote.read_bytes()).hexdigest():
                raise ValueError("Local and remote progress differ. Use a fresh runtime to resume the verified remote checkpoint.")
            if not local.exists():
                import shutil
                shutil.copyfile(remote, local)
    print("Resume files checked. Training has not started yet.")
else:
    print("Skipped checkpoint downloads.")


## Cell 8: Run exactly one stage (explicit GPU training only)

This is the only training cell. Each lesson uses a fresh Adam optimizer. Earlier task embeddings and chunk embeddings remain frozen after the shared lesson. Learning updates only the current task's BatchNorm running statistics. Unlearning freezes all task embeddings, chunk embeddings, and running statistics.

U3 uses the corrected generated-target-weight noise objective, including output scales and offsets, with protection of the seven retained tasks. It reads no task-3 training images during unlearning.

The before-unlearning checkpoint is preserved as the positive control. A failed readiness check still saves the completed unlearned model and measurements. It does not change gamma, beta, or run another experiment. Reports record epoch accuracies and measured runtime. Interrupted runs retain their last completed-lesson checkpoint.

In [ ]:
result = None
if MODE == "TRAIN":
    config = Config(dataset="tiny_imagenet", tasks=tuple(str(i) for i in range(20)), requests=(),
        backbone="resnet50", initialization="hyperfan_in", seed=SEED, epochs=5,
        batch_size=64, eval_batch_size=256, learning_rate=0.0001,
        forgetting_learning_rate=0.0001, beta=0.1, gamma=0.01,
        noise_samples=10, burn_in=100, device="cuda")
    def publish(event, output):
        if event == "before_unlearning":
            store.upload(output / "before_unlearning.pt", STAGE_PREFIX + "/before_unlearning.pt")
        if event == "request_complete":
            store.upload(output / "checkpoint.pt", STAGE_PREFIX + "/checkpoint.pt", overwrite=True)
        store.upload(output / "settings.json", STAGE_PREFIX + "/settings.json")
        store.upload(output / "report.json", STAGE_PREFIX + "/report.json", overwrite=True)
    result = run_setup_stage(config, tasks, OUTPUT, stage=STAGE, source=source,
        target=TARGET, retained=RETAINED, manifest_sha256=MANIFEST_HASH,
        code_version=CODE_VERSION, on_saved=publish, progress=True)
    print("Completed:", STAGE, "Seed:", SEED)
    print("Final validation accuracies:", result["final_accuracies"])
    if result["readiness"] is not None:
        print(json.dumps(result["readiness"], indent=2))
        if not result["readiness"]["eligible"]:
            print("Not eligible under the fixed pilot criteria. Preserve results and review before recovery.")
else:
    print("Skipped training. RUN_TRAINING remains off.")


## Cell 9: Verify final artifacts and release the GPU

A successful TRAIN stage verifies its remote final checkpoint and report before releasing Colab's GPU runtime. Save the printed R2 keys. You will use a new session for the next stage.

If an earlier cell failed, this cell does not release the runtime automatically. Inspect the error, preserve the local report if needed, and disconnect manually when finished. A storage failure must be fixed before considering the stage safely preserved.

After all three stages for seed 0, review the unlearning readiness checks and timings. Failed criteria need a documented decision; do not silently loosen them. Seeds 1 and 2 must also use fresh models from their own shared checkpoints. A separate notebook will implement recovery and the task-Y comparisons.

In [ ]:
if MODE == "TRAIN" and result is not None and result["status"] == "complete":
    from uncle.storage import sha256
    for name in ("checkpoint.pt", "report.json"):
        local = OUTPUT / name
        store.verify(STAGE_PREFIX + "/" + name, sha256(local), local.stat().st_size)
        print("Verified:", STAGE_PREFIX + "/" + name)
    if STAGE == "unlearned":
        local = OUTPUT / "before_unlearning.pt"
        store.verify(STAGE_PREFIX + "/before_unlearning.pt", sha256(local), local.stat().st_size)
        print("Verified positive control:", STAGE_PREFIX + "/before_unlearning.pt")
    if RELEASE_GPU_WHEN_DONE and IN_COLAB:
        from google.colab import runtime
        runtime.unassign()
else:
    print("No completed GPU stage to release.")
